# Analyze and record Bluesky Profiles (using db)

## init

### Imports

In [ ]:

from typing import  List
from IPython.display import  display

import psycopg2
from psycopg2.extras import RealDictCursor, execute_batch

from utils.schemas import StoredUserProfile, UserProfile, RankedUser

/Users/antoineestienne/GithubRepositories/ai-bootcamp-dev-repo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### clients

In [2]:
# Import additional dependencies for AI analysis
from openai import OpenAI
import instructor
import os
api_key=os.getenv("OPENAI_API_KEY")

# Initialize OpenAI client with instructor for structured outputs
openai_client = instructor.from_openai(OpenAI(api_key=api_key))

In [3]:
from atproto import Client
import os
from dotenv import load_dotenv

# Load environment variables (create a .env file with your credentials)
load_dotenv()

# Initialize the Bluesky client
bluesky_client = Client()

# Authenticate using your handle and app password
# Option 1: Using environment variables (recommended)
BLUESKY_HANDLE = os.getenv('BLUESKY_HANDLE')  # e.g., 'username.bsky.social'
BLUESKY_APP_PASSWORD = os.getenv('BLUESKY_APP_PASSWORD')  # Your app password

# Option 2: Hardcode (not recommended for production)
# BLUESKY_HANDLE = 'your-handle.bsky.social'
# BLUESKY_APP_PASSWORD = 'your-app-password'

# Login to Bluesky
bluesky_client.login(BLUESKY_HANDLE, BLUESKY_APP_PASSWORD)
print(f"✅ Successfully logged in as {BLUESKY_HANDLE}")

/Users/antoineestienne/GithubRepositories/ai-bootcamp-dev-repo/.venv/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'default' attribute with value None was provided to the `Field()` function, which has no effect in the context it was used. 'default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


✅ Successfully logged in as thedevguild.bsky.social


## Reusable functions

In [4]:
def insert_user_profile(user_profiles: List[StoredUserProfile]):
    """
    Insert a list of StoredUserProfile objects into the user_profiles.stored_user_profiles table.
    """
    if not user_profiles:
        print("No user profiles to insert.")
        return

    conn = None
    cursor = None
    try:
        # Connect to the database
        conn = psycopg2.connect(
            host="localhost",
            port=5433,
            database="langgraph_db",
            user="langgraph_user",
            password="langgraph_password"
        )
        conn.autocommit = True

        with conn.cursor(cursor_factory=RealDictCursor) as cursor:
            # The correct table (see user_profiles.sql) is user_profiles.stored_user_profiles
            insert_query = """
            INSERT INTO user_profiles.stored_user_profiles
            (handle, display_name, description, did, relevance_score, reasoning, follow_status, last_updated)
            VALUES (%(handle)s, %(display_name)s, %(description)s, %(did)s, %(relevance_score)s, %(reasoning)s, %(follow_status)s, to_timestamp(%(last_updated)s))
            ON CONFLICT (did) DO UPDATE SET
                handle = EXCLUDED.handle,
                display_name = EXCLUDED.display_name,
                description = EXCLUDED.description,
                relevance_score = EXCLUDED.relevance_score,
                reasoning = EXCLUDED.reasoning,
                follow_status = EXCLUDED.follow_status,
                last_updated = EXCLUDED.last_updated
            """

            # Transform user_profiles (which are Pydantic models) into dicts
            rows = [
                dict(user)
                if isinstance(user, dict)
                else user.dict()
                for user in user_profiles
            ]

            # Use execute_batch for efficient bulk upsert
            execute_batch(cursor, insert_query, rows, page_size=100)

            print(f"Successfully inserted/updated {len(rows)} records into user_profiles.stored_user_profiles")

    except psycopg2.Error as e:
        print(f"Database error: {e}")
        if conn:
            conn.rollback()
    except Exception as e:
        print(f"Error: {e}")
    finally:
        if conn:
            conn.close()

In [5]:
def analyze_user_with_ai(
    user: UserProfile, my_profile: UserProfile, openai_client: OpenAI
) -> RankedUser:
    """
    Use AI to analyze a single user and return their relevance to your profile.

    Args:
        user: UserProfile object to analyze
        my_profile: Your profile object
        openai_client: OpenAI client with instructor

    Returns:
        RankedUser object with relevance score and reasoning
    """
    if not user:
        print("⚠️ No user to analyze")
        return None

    print(f"\n🤖 Analyzing user @{user.handle} with AI...")

    # Prepare data for AI analysis
    user_data = {
        "handle": user.handle,
        "display_name": user.display_name or user.handle,
        "description": user.description or "",
        "did": user.did,
    }

    my_description = my_profile.description or ""
    my_display_name = my_profile.display_name or my_profile.handle

    # Create prompt for AI analysis
    analysis_prompt = f"""You are analyzing a Bluesky user to determine their relevance for someone to follow.

My Profile:
- Display Name: {my_display_name}
- Handle: {my_profile.handle}
- Description: {my_description}

User to analyze:
- Handle: @{user_data['handle']}
- Display Name: {user_data['display_name']}
- Description: {user_data['description']}
- DID: {user_data['did']}

Please analyze this user for relevance to my profile. Return:
- handle: The Bluesky handle
- display_name: Their display name
- description: Their profile description
- relevance_score: A score from 0.0 to 1.0 (1.0 = most relevant)
- reasoning: A brief explanation of why they're relevant
- did: The DID of the user

Focus on shared interests, professional connections, and content relevance."""

    try:
        response = openai_client.chat.completions.create(
            model="gpt-4.1",
            messages=[
                {
                    "role": "system",
                    "content": "You are an expert at analyzing social media profiles and finding relevant connections.",
                },
                {"role": "user", "content": analysis_prompt},
            ],
            response_model=RankedUser,
            temperature=0.3,
        )

        ranked_user = response
        print(
            f"✅ AI analysis complete! @{ranked_user.handle} ({ranked_user.display_name}): "
            f"{ranked_user.relevance_score:.2f} - {ranked_user.reasoning}"
        )

        return ranked_user

    except Exception as e:
        print(f"❌ Error during AI analysis: {e}")
        import traceback

        traceback.print_exc()
        return None


## Fetch and analyze some profiles

### Fetch profiles

In [22]:
from utils.profile_sources import fetch_followers_from_user, get_my_profile
from utils.profile_filters import filter_non_active_users


# Example: Get top 10 relevant followers from a user
target_username = 'hacktoberfest.com'  # Change this to any Bluesky username
max_number_of_followers_to_analyze = 10

# get all followers from a user
followers = fetch_followers_from_user(bluesky_client, target_username, max_number_of_followers_to_analyze)
followers=filter_non_active_users(bluesky_client, followers)

# print followers nicely
for follower in followers:
    print(f"Handle: {follower.handle}, Display Name: {follower.display_name}")

# get my profile
my_profile = get_my_profile(bluesky_client)


📊 Fetching followers of @hacktoberfest.com...
✅ Found profile: Hacktoberfest
   Followers: 680
✅ Fetched 10 followers

🔍 Filtering followers active in the last 30 days...
   Checking activity: 10/10...
✅ Found 7 active followers out of 10 checked

🔍 Filtering followers active in the last 30 days...
✅ Found 7 active followers out of 7 checked
Handle: thedevguild.bsky.social, Display Name: The Guild
Handle: joelamouche.bsky.social, Display Name: Antoine Estienne
Handle: ottoke.bsky.social, Display Name: Otto Kekäläinen
Handle: askthekaif.me, Display Name: kAiF
Handle: akshar-goyal.bsky.social, Display Name: Akshar Goyal
Handle: dmnchzl.dev, Display Name: Damien Chazoule
Handle: sapk.bsky.social, Display Name: Antoine G.
Getting my profile: thedevguild.bsky.social...
✅ My profile name: The Guild
✅ My profile description: On-chain reputation for developers.
🛠️ OSS & web3 builders
🏅 Peer-issued badges & attestations
⛓️ http://theguild.dev · 💬 https://discord.gg/axCqT23Xhj


### Analyze profile

In [7]:
analyzed_user=analyze_user_with_ai(followers[0], my_profile, openai_client)
analyzed_user


🤖 Analyzing user @thedevguild.bsky.social with AI...
✅ AI analysis complete! @thedevguild.bsky.social (The Guild): 1.00 - This user is an exact match to your profile, indicating they are either you or your official organization account. All interests, professional focus, and content are identical, making them maximally relevant.


RankedUser(handle='thedevguild.bsky.social', display_name='The Guild', description='On-chain reputation for developers.\n🛠️ OSS & web3 builders\n🏅 Peer-issued badges & attestations\n⛓️ http://theguild.dev · 💬 https://discord.gg/axCqT23Xhj', did='did:plc:a5nmb42bv7wuvjbkdlw2q3bs', relevance_score=1.0, reasoning='This user is an exact match to your profile, indicating they are either you or your official organization account. All interests, professional focus, and content are identical, making them maximally relevant.')

### follow profile

In [8]:
from utils.follow_profiles import follow_user
follow_user(followers[0], bluesky_client)

True

### insert profile into db

In [11]:
# add follow status and last_updated to the user in a new StoredUserProfile
import time
user_to_insert=StoredUserProfile(**analyzed_user.model_dump(), follow_status="followed", last_updated=int(time.time()))
insert_user_profile([user_to_insert])

Successfully inserted/updated 1 records into user_profiles.stored_user_profiles


/var/folders/hd/mwwk_4zs7blcj6tc0ycm9t5c0000gn/T/ipykernel_30982/3336513673.py:42: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  else user.dict()


### check that user is in the db

In [19]:
# this should check the status of the user in the db, return true if status is followed
def is_user_in_db(handle):
    conn = psycopg2.connect(
        host="localhost",
        port=5433,
        database="langgraph_db",
        user="langgraph_user",
        password="langgraph_password"   
    )
    try:
        cursor = conn.cursor(cursor_factory=RealDictCursor)
        query = """
        SELECT follow_status FROM user_profiles.stored_user_profiles WHERE handle = %s
        """
        cursor.execute(query, (handle,))
        result = cursor.fetchone()
        if result is None:
            return False  # User not in db, or no status to check
        # With RealDictCursor, result is a dict like {'follow_status': ...}
        return result.get("follow_status") == "followed"
    finally:
        conn.close()

# Example usage
print(is_user_in_db("thedevguild.bsky.social"))

True


In [20]:
print(is_user_in_db("hacktoberfest.com"))

False


In [ ]:
# process list of users
# see if already followed in db, if not followed and not already in db, analyze, follow and insert into db
# if followed, do nothing
# if already in db, do nothing
PERTINANCE_THRESHOLD=0.6
def process_users(users):
    for user in users:
        if not is_user_in_db(user.handle):
            print(f"User {user.handle} not followed and not in db, analyzing, following and inserting into db")
            analyzed_user=analyze_user_with_ai(user, my_profile, openai_client)
            follow_status="rejected"
            if analyzed_user.relevance_score >= PERTINANCE_THRESHOLD:
                follow_status="followed"
                follow_user(user, bluesky_client)
            user_to_insert=StoredUserProfile(**analyzed_user.model_dump(), follow_status=follow_status, last_updated=int(time.time()))
            insert_user_profile([user_to_insert])
        else:
            print(f"User {user.handle} already followed or in db")

# Example usage
process_users(followers)
print(f"Processed {len(followers)} users")

User thedevguild.bsky.social already followed or in db
User joelamouche.bsky.social already followed or in db
User ottoke.bsky.social already followed or in db
User askthekaif.me already followed or in db
User akshar-goyal.bsky.social already followed or in db
User dmnchzl.dev already followed or in db
User sapk.bsky.social already followed or in db
Processed 7 users
